In [1]:
import numpy as np
import pandas as pd

fars = pd.read_parquet("../data/fars_2000_2024.parquet")

In [2]:
#Drop the old columns not in the 2015-2024 original dataset
old_cols = ["road_fnc", "rel_junc", "traf_flo", "no_lanes", "sp_limit",
            "alignmnt", "profile", "pave_typ", "sur_cond", "tra_cont",
            "t_cont_f", "hit_run", "c_m_zone"]
fars = fars.drop(columns=old_cols)
fars

,state,county,month,day,hour,minute,ve_forms,persons,peds,nhs,...,not_minname,arr_hourname,arr_minname,hosp_hrname,hosp_mnname,cf1name,cf2name,cf3name,countyname,cityname
0,1,117,1,1,5,24,1,5,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,1,3,18,35,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,73,1,9,8,45,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,21,1,6,3,25,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,53,1,4,9,35,3,3,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
881051,56,9,12,14,15,45,2,2,0,1,...,50,4:00pm-4:59pm,19,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,CONVERSE (9),NOT APPLICABLE
881052,56,19,12,14,16,50,2,2,0,0,...,6,5:00pm-5:59pm,39,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,JOHNSON (19),NOT APPLICABLE
881053,56,31,12,18,10,45,1,1,0,0,...,34,10:00am-10:59am,43,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,PLATTE (31),NOT APPLICABLE
881054,56,21,12,28,17,43,1,2,1,0,...,46,5:00pm-5:59pm,49,6:00pm-6:59pm,8,NaN,NaN,NaN,LARAMIE (21),CHEYENNE


In [5]:
for col, low, high in [("hour", 0, 23), ("minute", 0, 59), ("day", 1, 31)]:
    print(col)
    print("  Missing:", fars[col].isna().sum())
    print("  Min:", fars[col].min(), " Max:", fars[col].max())
    print("  Values outside valid range:",
          (~fars[col].between(low, high) & fars[col].notna()).sum())

hour
  Missing: 0
  Min: 0  Max: 99
  Values outside valid range: 7167
minute
  Missing: 0
  Min: 0  Max: 99
  Values outside valid range: 7190
day
  Missing: 0
  Min: 1  Max: 99
  Values outside valid range: 78


In [7]:
# Change placeholder codes to missing values 
fars["hour"] = fars["hour"].where(fars["hour"].between(0, 23))
fars["minute"] = fars["minute"].where(fars["minute"].between(0, 59))
fars["day"] = fars["day"].where(fars["day"].between(1, 31))
fars

,state,county,month,day,hour,minute,ve_forms,persons,peds,nhs,...,not_minname,arr_hourname,arr_minname,hosp_hrname,hosp_mnname,cf1name,cf2name,cf3name,countyname,cityname
0,1,117,1,1.0,5.0,24.0,1,5,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,1,3.0,18.0,35.0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,73,1,9.0,8.0,45.0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,21,1,6.0,3.0,25.0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,53,1,4.0,9.0,35.0,3,3,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
881051,56,9,12,14.0,15.0,45.0,2,2,0,1,...,50,4:00pm-4:59pm,19,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,CONVERSE (9),NOT APPLICABLE
881052,56,19,12,14.0,16.0,50.0,2,2,0,0,...,6,5:00pm-5:59pm,39,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,JOHNSON (19),NOT APPLICABLE
881053,56,31,12,18.0,10.0,45.0,1,1,0,0,...,34,10:00am-10:59am,43,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,PLATTE (31),NOT APPLICABLE
881054,56,21,12,28.0,17.0,43.0,1,2,1,0,...,46,5:00pm-5:59pm,49,6:00pm-6:59pm,8,NaN,NaN,NaN,LARAMIE (21),CHEYENNE


In [23]:
for col, low, high in [("hour", 0, 23), ("minute", 0, 59), ("day", 1, 31)]:
    print(col)
    print("  Missing:", fars[col].isna().sum())
    print("  Min:", fars[col].min(), " Max:", fars[col].max())
    print("  Values outside valid range:",
          (~fars[col].between(low, high) & fars[col].notna()).sum())

hour
  Missing: 7167
  Min: 0.0  Max: 23.0
  Values outside valid range: 0
minute
  Missing: 7190
  Min: 0.0  Max: 59.0
  Values outside valid range: 0
day
  Missing: 78
  Min: 1.0  Max: 31.0
  Values outside valid range: 0


In [11]:
fars["crash_date"] = pd.to_datetime(
    dict(year=fars["year"], month=fars["month"], day=fars["day"]),
    errors="coerce",
)
 
def time_of_day(hour):
    if pd.isna(hour):
        return np.nan
    if 0 <= hour < 6:
        return "Overnight"   # 12am - 5:59am
    if 6 <= hour < 12:
        return "Morning"     # 6am - 11:59am
    if 12 <= hour < 18:
        return "Afternoon"   # 12pm - 5:59pm
    return "Evening"         # 6pm - 11:59pm
 
fars["time_of_day"] = fars["hour"].apply(time_of_day)
fars

,state,county,month,day,hour,minute,ve_forms,persons,peds,nhs,...,arr_minname,hosp_hrname,hosp_mnname,cf1name,cf2name,cf3name,countyname,cityname,crash_date,time_of_day
0,1,117,1,1.0,5.0,24.0,1,5,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,Overnight
1,1,1,1,3.0,18.0,35.0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-03,Evening
2,1,73,1,9.0,8.0,45.0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-09,Morning
3,1,21,1,6.0,3.0,25.0,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-06,Overnight
4,1,53,1,4.0,9.0,35.0,3,3,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-04,Morning
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
881051,56,9,12,14.0,15.0,45.0,2,2,0,1,...,19,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,CONVERSE (9),NOT APPLICABLE,2024-12-14,Afternoon
881052,56,19,12,14.0,16.0,50.0,2,2,0,0,...,39,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,JOHNSON (19),NOT APPLICABLE,2024-12-14,Afternoon
881053,56,31,12,18.0,10.0,45.0,1,1,0,0,...,43,Not Applicable (Not Transported),Not Applicable (Not Transported),NaN,NaN,NaN,PLATTE (31),NOT APPLICABLE,2024-12-18,Morning
881054,56,21,12,28.0,17.0,43.0,1,2,1,0,...,49,6:00pm-6:59pm,8,NaN,NaN,NaN,LARAMIE (21),CHEYENNE,2024-12-28,Afternoon


In [13]:
fars.to_parquet("../data/fars_2000_2024_clean.parquet", index=False)